# 02 · Long Time-Series Extraction — Landsat 5/8/9 NDSI (2000 – 2026)

We assemble a single, sensor-merged NDSI ImageCollection spanning the full study window from Google Earth Engine, then reduce it to a regional-mean time-series across the Khulna – Satkhira – Bagerhat domain. The output is cached as a CSV in `data/` so downstream notebooks (03 – 06) can avoid the ~30-minute GEE pull.

**Outputs:**
- `data/ndsi_timeseries_2000_2026.csv` — regional-mean NDSI per scene
- `figures/fig_long_timeseries.png` — full record with cyclone overlays

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

from src.config import (
    ANALYSIS_START_DATE, ANALYSIS_END_DATE,
    DATA_DIR, FIGURES_DIR,
)
from src.gee_ndsi import initialize_ee, build_ndsi_collection, extract_regional_mean
from src.seasonality import rolling_mean
from src.visualization import plot_timeseries

In [ ]:
initialize_ee()
print(f'Building NDSI collection: {ANALYSIS_START_DATE} → {ANALYSIS_END_DATE}')
print('Platforms: Landsat 5, 8, 9 (Collection 2 Tier 1 L2)')

## 1 · Build the merged collection

GEE evaluates lazily — building the collection is instant; the cost is paid in the next cell when we reduce.

In [ ]:
coll = build_ndsi_collection(
    start=ANALYSIS_START_DATE,
    end=ANALYSIS_END_DATE,
    platforms=('L5', 'L8', 'L9'),
)
n_scenes = coll.size().getInfo()
print(f'Cloud-free scenes (cloud cover < 15 %): {n_scenes}')

## 2 · Reduce to regional-mean NDSI per scene

This is the expensive step — typically 15–30 minutes for the full 26-year archive. The result is cached to CSV so you only need to do this once.

In [ ]:
cache_path = DATA_DIR / 'ndsi_timeseries_2000_2026.csv'

if cache_path.exists():
    import pandas as pd
    ts = pd.read_csv(cache_path, parse_dates=['date'])
    print(f'Loaded cached time-series ({len(ts)} rows) from {cache_path}')
else:
    print('Extracting regional means from GEE — this can take 15–30 min...')
    ts = extract_regional_mean(coll, scale=30)
    ts.to_csv(cache_path, index=False)
    print(f'Cached {len(ts)} rows to {cache_path}')

ts.head()

## 3 · Summary statistics

In [ ]:
print(f'Date range:       {ts["date"].min().date()} → {ts["date"].max().date()}')
print(f'Total scenes:     {len(ts)}')
print(f'NDSI range:       {ts["ndsi_mean"].min():.4f} → {ts["ndsi_mean"].max():.4f}')
print(f'NDSI mean ± std:  {ts["ndsi_mean"].mean():.4f} ± {ts["ndsi_mean"].std():.4f}')
print(f'Scenes per year:  {len(ts) / (ts["date"].dt.year.nunique()):.1f} (avg)')

## 4 · Headline time-series plot (with cyclone overlays)

In [ ]:
smoothed = rolling_mean(ts, value_col='ndsi_mean', window_months=12)

fig_path = FIGURES_DIR / 'fig_long_timeseries.png'
plot_timeseries(
    ts,
    value_col='ndsi_mean',
    rolling=smoothed,
    annotate_cyclones=True,
    save_to=fig_path,
)
print(f'Saved: {fig_path}')

---

**Reconstruction complete.** The cached CSV is the input for every subsequent notebook:

- **03** — Cyclone-event attribution (pre / post landfall windows + bootstrap CIs)
- **04** — Seasonal climatology + August anomaly
- **05** — Random Forest training
- **06** — 2050 projection + hotspot identification